# FEATURE ABLATION

In [ ]:
import pandas as pd
import torch
import os
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score

# Import project-specific modules
from data_preparation import prepare_dataset, get_loss_weights
from graph_construction import build_graph
from train_and_evaluate import train_and_evaluate
from models import GoldenTransformer
from resources import (
    DATA_PATH,
    DEVICE,
    OUTPUT_TABLES_PATH,
    OUTPUT_FIGURES_PATH
)

def set_seed(seed):
    """Enforces reproducible behavior across all random libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # Guarantees reproducible, deterministic convolution algorithms at a slight speed cost
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# 1. SETUP & DATA LOADING
print("Initializing Feature Ablation Study...")
x, y, pos_combined, pos_spatial, pos_temporal, group_ids = prepare_dataset(DATA_PATH, return_group_ids=True)
class_weights = get_loss_weights(y=y)

# Load hyperparameter configurations from previous study
best_transformer_df = pd.read_csv(os.path.join(OUTPUT_TABLES_PATH, "best_transformer_df.csv"))
best_row = best_transformer_df.sort_values(by='F1', ascending=False).iloc[0]

BEST_K = int(best_row['K']) 
BEST_HEADS = int(best_row['Heads'])

# Define features and experiment parameters
feature_names = ['Log-Area', 'Latitude', 'Longitude', 'Days Since Start']
seeds = [111, 222, 444, 555, 666, 777, 888, 999, 1111, 2222]
NUM_RUNS = len(seeds)

# Prepare the full tensor
log_area_t = x.to(DEVICE) if x.dim() == 2 else x.unsqueeze(1).to(DEVICE)
x_full = torch.cat([
    log_area_t, 
    pos_spatial.to(DEVICE), 
    pos_temporal.to(DEVICE)
], dim=1)


# --- 2. EVALUATE CONTROL BASELINE (ALL FEATURES) ---
print(f"\nEvaluating Baseline Performance (All Features) over {NUM_RUNS} runs...")
baseline_scores = []

for run in range(NUM_RUNS):
    set_seed(seeds[run])
    
    # Rebuild complete graph with all features
    data_full = build_graph('multirelational', BEST_K, pos_spatial, pos_temporal, pos_combined, x_full, y, group_ids=group_ids).to(DEVICE)
    
    model = GoldenTransformer(
        input_dim=data_full.x.size(1), 
        hidden_dim=16, 
        edge_dim=data_full.edge_attr.size(1), 
        num_heads=BEST_HEADS
    ).to(DEVICE)

    f1, _, _ = train_and_evaluate(model, data_full, DEVICE, class_weights, max_epochs=500, patience=30)
    baseline_scores.append(f1)

MEAN_BASELINE_F1 = np.mean(baseline_scores)
STD_BASELINE_F1 = np.std(baseline_scores)
print(f"True Baseline F1: {MEAN_BASELINE_F1:.4f} +/- {STD_BASELINE_F1:.4f}")
print("-" * 60)


# --- 3. FEATURE ABLATION (LEAVE-ONE-OUT) ---
ablation_results = []

for i, f_name in enumerate(feature_names):
    print(f"Ablating Feature: {f_name}...")
    
    # Create mask to exclude one feature
    mask = torch.ones(x_full.size(1), dtype=torch.bool)
    mask[i] = False
    x_ablated = x_full[:, mask]

    run_scores = []
    for run in range(NUM_RUNS):
        set_seed(seeds[run])  # Syncs weight init and data splits to mirror baseline setup exactly
        
        # Rebuild graph for each run with ablated node features
        data_ablated = build_graph('multirelational', BEST_K, pos_spatial, pos_temporal, pos_combined, x_ablated, y, group_ids=group_ids).to(DEVICE)
        
        # Re-initialize model with adjusted input dimension (3 features instead of 4)
        model = GoldenTransformer(
            input_dim=data_ablated.x.size(1), 
            hidden_dim=16, 
            edge_dim=data_ablated.edge_attr.size(1), 
            num_heads=BEST_HEADS
        ).to(DEVICE)

        f1, _, _ = train_and_evaluate(model, data_ablated, DEVICE, class_weights, max_epochs=500, patience=30)
        run_scores.append(f1)

    mean_f1 = np.mean(run_scores)
    std_f1 = np.std(run_scores)
    
    # Compare 10-run mean directly against 10-run baseline mean
    f1_drop = MEAN_BASELINE_F1 - mean_f1
    relative_importance = (f1_drop / MEAN_BASELINE_F1) * 100

    print(f"  -> Mean F1: {mean_f1:.4f} +/- {std_f1:.4f} | Drop: {f1_drop:.4f} ({relative_importance:+.2f}%)")
    
    ablation_results.append({
        'Feature_Ablated': f_name,
        'Mean_F1': mean_f1,
        'Std_F1': std_f1,
        'F1_Drop': f1_drop,
        'Importance_Percent': relative_importance
    })

# Convert to DataFrame and save output
df_results = pd.DataFrame(ablation_results)
df_results.to_csv(os.path.join(OUTPUT_TABLES_PATH, "feature_ablation_results.csv"), index=False)
print(f"\nAblation study completed. Results saved to {OUTPUT_TABLES_PATH}")

In [ ]:
# 3. REPORTING & VISUALIZATION
ablation_report_df = pd.DataFrame(ablation_results).sort_values(by='Importance_Percent', ascending=True)
ablation_report_df.to_csv(os.path.join(OUTPUT_TABLES_PATH, "feature_ablation_results.csv"), index=False)

# ==========================================
# GLOBAL PAPER-READY STYLING
# ==========================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 18,
    'axes.labelsize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'axes.linewidth': 1.5,
    'figure.dpi': 300,
    'savefig.bbox': 'tight'
})
sns.set_theme(style="whitegrid", font="serif")

# Plotting the Importance Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))

# Use a professional sequential palette or a single color to avoid "rainbow" effects
sns.barplot(
    data=ablation_report_df, 
    x='Importance_Percent', 
    y='Feature_Ablated', 
    palette='Blues_d',   # Professional blue gradient
    edgecolor='black',
    linewidth=1.2,
    ax=ax
)

# Standard Academic Labels
ax.set_xlabel("Relative Importance (% Drop in Macro F1)", fontweight='bold')
ax.set_ylabel("Ablated Feature", fontweight='bold')

# Add percentage labels at the end of bars for immediate readability
for i, p in enumerate(ax.patches):
    width = p.get_width()
    ax.text(width + (max(ablation_report_df['Importance_Percent']) * 0.05), 
            p.get_y() + p.get_height()/2, 
            f'{width:+.2f}%', 
            va='center', fontsize=14, fontweight='bold')

# Ensure the X-axis starts at 0 and has enough padding for labels
ax.set_xlim(0, max(ablation_report_df['Importance_Percent']) * 1.25)

plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, "feature_ablation_importance.png"))
plt.show()

print("\n--- Feature Ablation Study Complete ---")
print(ablation_report_df[['Feature_Ablated', 'F1_Drop', 'Importance_Percent']].sort_values(by='Importance_Percent', ascending=False).to_string(index=False))